# Movie_Feature (Supporting class)

In [ ]:
import numpy as np
import scipy.sparse as sparse
from sklearn.decomposition import TruncatedSVD

class MovieFeatureExtractor:
    """
    Lớp trích xuất đặc trưng nội dung phim (Content-Based Filtering)
    sử dụng Latent Semantic Analysis (LSA) trên ma trận thưa TF-IDF.
    """
    def __init__(self, k_dimensions=100, random_state=42):
        """
        Khởi tạo Extractor.
        
        Parameters:
        - k_dimensions: Số lượng chủ đề ẩn (Latent topics) muốn giữ lại.
        - random_state: Seed để tái tạo lại kết quả.
        """
        self.k_dimensions = k_dimensions
        self.random_state = random_state
        self.variance_retained = 0.0
        
        # Khởi tạo mô hình TruncatedSVD
        self.svd_model = TruncatedSVD(
            n_components=self.k_dimensions, 
            algorithm='randomized', 
            random_state=self.random_state
        )

    def fit(self, tfidf_mat):
        """
        Huấn luyện mô hình để tìm ra không gian vector ẩn (Latent space).
        
        Parameters:
        - tfidf_mat: Ma trận scipy.sparse cần để train.
        
        Returns:
        - self: Trả về chính object hiện tại.
        """
        print(f"[*] Đang huấn luyện mô hình SVD (Fit) với k = {self.k_dimensions}...")
        
        # Fit ma trận TF-IDF để học U, Sigma, V^T
        self.svd_model.fit(tfidf_mat)
        
        # Phân tích phương sai (Độ tin cậy của mô hình)
        self.variance_retained = np.sum(self.svd_model.explained_variance_ratio_) * 100
        
        print(f"[+] Huấn luyện hoàn tất!")
        print(f"[+] Tổng lượng thông tin được giữ lại (Explained Variance): {self.variance_retained:.2f}%")
        
        # Cảnh báo học thuật
        if self.variance_retained < 60:
            print("[!] Cảnh báo: Lượng thông tin giữ lại < 60%. Bạn nên cân nhắc tăng k_dimensions.")
            
        return self

    def transform(self, tfidf_mat):
        """
        Chiếu ma trận TF-IDF vào không gian ẩn đã được học từ bước fit.
        
        Parameters:
        - tfidf_mat: Ma trận scipy.sparse cần biến đổi.
        
        Returns:
        - item_content_profiles: Ma trận đặc trưng nội dung của video (Dense array)
        """
        print(f"[*] Đang biến đổi dữ liệu (Transform)...")
        
        # Transform dữ liệu
        item_content_profiles = self.svd_model.transform(tfidf_mat)
        
        print(f"[+] Kích thước Item Content Profiles: {item_content_profiles.shape}")
        return item_content_profiles

    def fit_transform(self, tfidf_mat):
        """
        Kết hợp fit và transform trong 1 bước (tối ưu hóa hiệu năng tính toán).
        """
        self.fit(tfidf_mat)
        return self.transform(tfidf_mat)


# ==========================================
# --- THỰC THI CHƯƠNG TRÌNH ---
# ==========================================

# Giả lập tạo một ma trận thưa TF-IDF ngẫu nhiên (46625 movies x 10000 keywords)
# Trong thực tế, bạn sẽ dùng: tfidf_mat = sparse.load_npz(npz_path)
print("[-] Đang tạo ma trận TF-IDF giả lập...")
tfidf_mat = tfidf_mat

# 1. Khởi tạo class
extractor = MovieFeatureExtractor(k_dimensions=100)

# 2. Fit và Transform
# Cách 1: Chạy fit riêng và transform riêng (Hữu ích khi train xong muốn lưu model lại)


# Cách 2: Chạy fit_transform cùng lúc cho tập train
movie_factors_matrix = extractor.fit_transform(tfidf_mat)
print(f"3. Kích thước ma trận trọng số đã học: {extractor.svd_model.components_.shape}")

# 3. Kiểm tra kết quả
non_zero_count = np.count_nonzero(movie_factors_matrix)
total_elements = movie_factors_matrix.size

print("-" * 40)
print(f"[+] Số lượng phần tử khác 0: {non_zero_count}") 
print(f"[+] Tỷ lệ phần tử khác 0 (Sparsity check): {(non_zero_count / total_elements) * 100:.4f}%")

# Lưu trữ artifact để phục vụ việc huấn luyện Meta-learner
# np.save('item_content_factors_k100.npy', item_cbf_factors)

# Project_root

In [18]:
# Cell này giúp xuất DataFrame từ TF-IDF artifacts và giải thích giới hạn khôi phục dữ liệu gốc.
from pathlib import Path
import pickle
import pandas as pd
from scipy import sparse

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data" / "processed").exists()),
    None,
 )
if project_root is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed từ thư mục hiện tại.")

processed_dir = project_root / "data" / "processed"



# Class build data for training and testing

In [19]:
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD

class RecommenderDatasetBuilder:
    def __init__(self, ratings_npz_path, ratings_pkl_path, movies_npz_path, movies_pkl_path, k_dimensions=100, random_state=42):
        """
        Khởi tạo class và load toàn bộ dữ liệu từ các đường dẫn file.
        """
        print("Đang load dữ liệu...")
        
        # 1. Load file Pickle (.pkl)
        with open(ratings_pkl_path, 'rb') as f:
            self.collab_pkl_data = pickle.load(f)
            
        with open(movies_pkl_path, 'rb') as f:
            self.tfidf_pkl_data = pickle.load(f)

        # 2. Load file NPZ (.npz)
        # Ma trận ratings lưu dạng sparse (scipy)
        self.collab_mat = sp.load_npz(ratings_npz_path)

        # movies_npz_path là ma trận TF-IDF sparse, KHÔNG phải movie_factors_matrix đã nén bằng np.savez
        # Vì vậy cần chiếu qua SVD để tạo movie_factors_matrix đúng như MovieFeatureExtractor.
        self.tfidf_mat = sp.load_npz(movies_npz_path)

        # Đảm bảo k hợp lệ cho TruncatedSVD: 1 <= k < min(n_samples, n_features)
        max_k = min(self.tfidf_mat.shape) - 1
        if max_k < 1:
            raise ValueError(f"movies_npz_path có shape không hợp lệ cho SVD: {self.tfidf_mat.shape}")
        effective_k = min(k_dimensions, max_k)

        svd = TruncatedSVD(
            n_components=effective_k,
            algorithm='randomized',
            random_state=random_state,
        )
        self.movie_factors_matrix = svd.fit_transform(self.tfidf_mat)

        print(f"Shape của ma trận phim (movie factors): {self.movie_factors_matrix.shape}")

        # 3. Khởi tạo các dictionary mapping
        self.userid_to_index_collab = self.collab_pkl_data['user_mapping']
        self.index_to_userid_collab = {v: k for k, v in self.userid_to_index_collab.items()}

        self.movieid_to_index_collab = self.collab_pkl_data['movie_mapping']
        self.index_to_movieid_collab = {v: k for k, v in self.movieid_to_index_collab.items()}

        self.movieid_to_index_tfidf = self.tfidf_pkl_data['movie_mapping']
        self.index_to_movieid_tfidf = {v: k for k, v in self.movieid_to_index_tfidf.items()}
        
        # Trích xuất dữ liệu từ sparse matrix
        self.rows, self.cols = self.collab_mat.nonzero()
        self.values = self.collab_mat.data

    def _calculate_user_profiles(self):
        """
        BƯỚC 1: TÍNH VECTOR TRUNG BÌNH CHO TỪNG USER
        """
        print("Đang tính toán vector trung bình cho từng User...")
        user_movie_vectors = {}

        for user_idx_in_collab, movie_idx_in_collab in zip(self.rows, self.cols):
            user_id = self.index_to_userid_collab.get(user_idx_in_collab, "Unknown User")
            movie_id = self.index_to_movieid_collab.get(movie_idx_in_collab, "Unknown Movie")
            movie_idx_in_tfidf = self.movieid_to_index_tfidf.get(movie_id, None)

            if movie_idx_in_tfidf is None:
                continue  # Bỏ qua nếu không map được

            # Lấy vector phim và flatten
            movie_vector = self.movie_factors_matrix[movie_idx_in_tfidf].flatten()
            
            # Lưu vào danh sách các phim của user này
            if user_id not in user_movie_vectors:
                user_movie_vectors[user_id] = []
            user_movie_vectors[user_id].append(movie_vector)

        # Tính trung bình các vector phim cho mỗi user
        user_avg_profiles = {}
        for user_id, vectors in user_movie_vectors.items():
            user_avg_profiles[user_id] = np.mean(vectors, axis=0)
            
        return user_avg_profiles

    def _generate_dataset(self, user_avg_profiles):
        """
        BƯỚC 2: TẠO MA TRẬN X VÀ y CUỐI CÙNG
        """
        print("Đang tạo dataset (X, y)...")
        X_list = []
        y_list = []

        for user_idx_in_collab, movie_idx_in_collab, rating in zip(self.rows, self.cols, self.values):
            user_id = self.index_to_userid_collab.get(user_idx_in_collab, "Unknown User")
            movie_id = self.index_to_movieid_collab.get(movie_idx_in_collab, "Unknown Movie")
            movie_idx_in_tfidf = self.movieid_to_index_tfidf.get(movie_id, None)

            if movie_idx_in_tfidf is None:
                continue

            # 1. Lấy vector phim
            movie_vector = self.movie_factors_matrix[movie_idx_in_tfidf].flatten()
            
            # 2. Lấy vector user trung bình
            user_vector = user_avg_profiles[user_id]
            
            # 3. Nối (concatenate) hai vector
            combined_features = np.concatenate((user_vector, movie_vector))
            
            X_list.append(combined_features)
            y_list.append(rating)

        X = np.array(X_list)
        y = np.array(y_list)
        
        return X, y

    def build(self):
        """
        Hàm chính (Public method) để gọi và thực thi toàn bộ quy trình
        """
        user_avg_profiles = self._calculate_user_profiles()
        X, y = self._generate_dataset(user_avg_profiles)
        
        print("\n--- HOÀN TẤT ---")
        print(f"Số lượng mẫu đã tạo: {len(X)}")
        if len(X) > 0:
            print(f"Kích thước của ma trận X: {X.shape}")
            print(f"Kích thước của vector y: {y.shape}")
            # In ra một phần nhỏ để kiểm tra thay vì cả mảng to
            print(f"Mẫu X đầu tiên (5 phần tử đầu): {X[0][:5]} ...")
            
        return X, y

In [ ]:
# Khai báo đường dẫn đến 4 file của bạn
TRAIN_RATINGS_NPZ = processed_dir / "train_warm_ratings_matrix.npz"
TRAIN_RATINGS_PKL = processed_dir / "train_warm_ratings_mapping.pkl"
TRAIN_MOVIES_NPZ = processed_dir / "train_warm_movies_matrix.npz"
TRAIN_MOVIES_PKL = processed_dir / "train_warm_movies_mapping.pkl"

# Khởi tạo class
dataset_builder = RecommenderDatasetBuilder(
    ratings_npz_path=TRAIN_RATINGS_NPZ,
    ratings_pkl_path=TRAIN_RATINGS_PKL,
    movies_npz_path=TRAIN_MOVIES_NPZ,
    movies_pkl_path=TRAIN_MOVIES_PKL
)

# Chạy quy trình và lấy kết quả X, y
X_train, y_train = dataset_builder.build()

# Bây giờ bạn có thể dùng X, y để train model (ví dụ: RandomForest, XGBoost, Neural Network...)


Đang load dữ liệu...
Shape của ma trận phim (movie factors): (5151, 100)
Đang tính toán vector trung bình cho từng User...
Đang tạo dataset (X, y)...

--- HOÀN TẤT ---
Số lượng mẫu đã tạo: 8582160
Kích thước của ma trận X: (8582160, 200)
Kích thước của vector y: (8582160,)
Mẫu X đầu tiên (5 phần tử đầu): [ 0.12404398 -0.00380396 -0.00836936  0.00813288 -0.01039773] ...


In [23]:
TEST_RATINGS_NPZ = processed_dir / "test_warm_ratings_matrix.npz"
TEST_RATINGS_PKL = processed_dir / "test_warm_ratings_mapping.pkl"
MOVIES_NPZ = processed_dir / "train_warm_movies_matrix.npz"
MOVIES_PKL = processed_dir / "train_warm_movies_mapping.pkl"

dataset_builder_test = RecommenderDatasetBuilder(
    ratings_npz_path=TEST_RATINGS_NPZ,
    ratings_pkl_path=TEST_RATINGS_PKL,
    movies_npz_path=MOVIES_NPZ,
    movies_pkl_path=MOVIES_PKL
)
X_test, y_test = dataset_builder_test.build()

Đang load dữ liệu...
Shape của ma trận phim (movie factors): (5151, 100)
Đang tính toán vector trung bình cho từng User...
Đang tạo dataset (X, y)...

--- HOÀN TẤT ---
Số lượng mẫu đã tạo: 2145434
Kích thước của ma trận X: (2145434, 200)
Kích thước của vector y: (2145434,)
Mẫu X đầu tiên (5 phần tử đầu): [ 0.12283383  0.0044892  -0.01068475  0.00036507 -0.00925839] ...


In [29]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(8582160, 200) (8582160,)
(2145434, 200) (2145434,)


# Trainning

In [26]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# 0. Chuẩn bị dữ liệu (Giả sử bạn đã có X và y)
# LỜI KHUYÊN: Ép kiểu X về float32 để giảm một nửa dung lượng RAM 
# (từ ~9GB xuống còn ~4.5GB)
# ---------------------------------------------------------
# X = np.array(X_list, dtype=np.float32) 
# y = np.array(y_list, dtype=np.float32)

print("1. Đang chia tập dữ liệu...")

print("2. Đang tạo LightGBM Dataset...")
# Khởi tạo Dataset của LightGBM. 
# free_raw_data=True giúp giải phóng RAM từ mảng NumPy ban đầu sau khi tạo Dataset
train_data = lgb.Dataset(X_train, label=y_train, free_raw_data=True)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data, free_raw_data=True)

print("3. Thiết lập thông số (Hyperparameters)...")
params = {
    'objective': 'regression',       # Bài toán dự đoán số thực (điểm rating)
    'metric': 'rmse',                # Đo lường sai số bằng Root Mean Squared Error
    'boosting_type': 'gbdt',         # Gradient Boosting Decision Tree truyền thống
    'learning_rate': 0.01,           # Tốc độ học (nhỏ thì mô hình học kỹ hơn nhưng lâu hơn)
    'num_leaves': 63,                # Số lá tối đa trên mỗi cây (Tăng lên 63 vì data bạn rất lớn)
    'max_depth': -1,                 # Không giới hạn độ sâu (LightGBM tự ưu tiên phát triển theo chiều sâu của lá)
    'feature_fraction': 0.8,         # Mỗi cây chỉ lấy ngẫu nhiên 80% trong số 100 features để chống overfit
    'bagging_fraction': 0.8,         # Lấy ngẫu nhiên 80% data cho mỗi vòng lặp
    'bagging_freq': 5,               # Tần suất thực hiện bagging
    'n_jobs': -1,                    # Tận dụng TẤT CẢ các luồng CPU đang có
    # 'device': 'gpu'                # BỎ COMMENT dòng này nếu máy bạn có GPU đã cài đặt CUDA
}

print("4. Bắt đầu huấn luyện...")
# Sử dụng API mới của LightGBM với callbacks
model = lgb.train(
    params,
    train_data,
    num_boost_round=100,            # Cho phép tạo tối đa 5000 cây
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        # Dừng quá trình train nếu sau 50 cây mà điểm trên tập Validation không giảm
        lgb.early_stopping(stopping_rounds=50), 
        # In kết quả ra màn hình sau mỗi 100 cây
        lgb.log_evaluation(period=100)          
    ]
)

print("5. Đánh giá mô hình trên tập Validation...")
# Dự đoán trên tập Validation
y_pred = model.predict(X_test)

# Tính toán sai số
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- KẾT QUẢ ---")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")

# (Tùy chọn) Xem mức độ quan trọng của các Features
# Cho biết trong 100 features của bạn, thông tin nào đóng góp nhiều nhất vào việc đoán điểm
# import matplotlib.pyplot as plt
# lgb.plot_importance(model, max_num_features=20, figsize=(10, 6))
# plt.show()

1. Đang chia tập dữ liệu...
2. Đang tạo LightGBM Dataset...
3. Thiết lập thông số (Hyperparameters)...
4. Bắt đầu huấn luyện...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 4.480489 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 50980
[LightGBM] [Info] Number of data points in the train set: 8582160, number of used features: 200
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[17]	train's rmse: 0.869407	valid's rmse: 0.838685
5. Đánh giá mô hình trên tập Validation...

--- KẾT QUẢ ---
RMSE (Root Mean Squared Error): 0.8387
MAE (Mean Absolute Error): 0.6440


# XGBoost Training

In [30]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# 0. Chuẩn bị dữ liệu (Giả sử bạn đã có X và y)
# LỜI KHUYÊN: Ép kiểu X về float32 để giảm một nửa dung lượng RAM 
# (từ ~9GB xuống còn ~4.5GB)
# ---------------------------------------------------------
# X = np.array(X_list, dtype=np.float32) 
# y = np.array(y_list, dtype=np.float32)

print("1. Đang chia tập dữ liệu...")

print("2. Đang tạo XGBoost DMatrix...")
# Khởi tạo DMatrix của XGBoost cho tập train và validation
train_matrix = xgb.DMatrix(X_train, label=y_train)
valid_matrix = xgb.DMatrix(X_test, label=y_test)

print("3. Thiết lập thông số (Hyperparameters)...")
params = {
    'objective': 'reg:squarederror',       # Bài toán dự đoán số thực (điểm rating)
    'metric': 'rmse',                      # Đo lường sai số bằng Root Mean Squared Error
    'booster': 'gbtree',                   # Sử dụng cây quyết định
    'eta': 0.01,                           # Tốc độ học (nhỏ thì mô hình học kỹ hơn nhưng lâu hơn)
    'max_depth': 6,                        # Độ sâu tối đa của cây
    'subsample': 0.8,                      # Lấy ngẫu nhiên 80% data cho mỗi cây
    'colsample_bytree': 0.8,               # Lấy ngẫu nhiên 80% features cho mỗi cây (chống overfit)
    'lambda': 1.0,                         # Hệ số regularization L2 (tránh overfitting)
    'alpha': 0.0,                          # Hệ số regularization L1
    'tree_method': 'auto',                 # Phương thức xây dựng cây (auto = tự chọn tối ưu)
    # 'tree_method': 'gpu_hist'             # BỎ COMMENT dòng này nếu máy bạn có GPU đã cài đặt CUDA
    'verbosity': 1                         # In log chi tiết
}

print("4. Bắt đầu huấn luyện...")
# Sử dụng XGBoost train API
evals_result = {}
model = xgb.train(
    params,
    train_matrix,
    num_boost_round=100,                   # Cho phép tạo tối đa 100 cây
    evals=[(train_matrix, 'train'), (valid_matrix, 'valid')],
    evals_result=evals_result,
    early_stopping_rounds=50,              # Dừng nếu sau 50 cây điểm Validation không giảm
    verbose_eval=10                        # In kết quả ra màn hình sau mỗi 10 cây
)

print("5. Đánh giá mô hình trên tập Validation...")
# Dự đoán trên tập Validation
y_pred = model.predict(valid_matrix)

# Tính toán sai số
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- KẾT QUẢ ---")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")

# (Tùy chọn) Xem mức độ quan trọng của các Features
# import matplotlib.pyplot as plt
# xgb.plot_importance(model, max_num_features=20, figsize=(10, 6))
# plt.show()

1. Đang chia tập dữ liệu...
2. Đang tạo XGBoost DMatrix...
3. Thiết lập thông số (Hyperparameters)...
4. Bắt đầu huấn luyện...


e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [16:44:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "metric" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:0.87082	valid-rmse:0.83876
[10]	train-rmse:0.86990	valid-rmse:0.83871
[20]	train-rmse:0.86911	valid-rmse:0.83870
[30]	train-rmse:0.86844	valid-rmse:0.83874
[40]	train-rmse:0.86784	valid-rmse:0.83882
[50]	train-rmse:0.86733	valid-rmse:0.83891
[60]	train-rmse:0.86687	valid-rmse:0.83901
[65]	train-rmse:0.86666	valid-rmse:0.83908
5. Đánh giá mô hình trên tập Validation...

--- KẾT QUẢ ---
RMSE (Root Mean Squared Error): 0.8391
MAE (Mean Absolute Error): 0.6447


# Stochastic Gradient Descent (SGD) Training

In [32]:
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# 0. Chuẩn bị dữ liệu (Giả sử bạn đã có X và y)
# LỜI KHUYÊN: Ép kiểu X về float32 để giảm một nửa dung lượng RAM 
# (từ ~9GB xuống còn ~4.5GB)
# ---------------------------------------------------------
# X = np.array(X_list, dtype=np.float32) 
# y = np.array(y_list, dtype=np.float32)

print("1. Tạo mô hình SGD Regressor...")

print("2. Thiết lập thông số (Hyperparameters)...")
sgd_model = SGDRegressor(
    loss='squared_error',                  # Sử dụng squared error loss function
    penalty='l2',                          # Regularization L2 (Ridge)
    alpha=0.0001,                          # Hệ số regularization
    max_iter=1000,                         # Số lượng epochs tối đa
    learning_rate='optimal',               # Adaptive learning rate
    eta0=0.01,                             # Learning rate ban đầu
    validation_fraction=0.1,               # 10% dữ liệu dùng làm validation
    early_stopping=True,                   # Dừng sớm nếu validation score không cải thiện
    tol=1e-3,                              # Tolerance cho early stopping
    random_state=42,
    verbose=0
)

print("3. Bắt đầu huấn luyện...")
# Fit model trên tập training
sgd_model.fit(X_train, y_train)

print("4. Đánh giá mô hình trên tập Validation...")
# Dự đoán trên tập Validation
y_pred = sgd_model.predict(X_test)

# Tính toán sai số
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

# Tính toán R² score
r2_score = sgd_model.score(X_test, y_test)

print(f"\n--- KẾT QUẢ ---")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R² Score: {r2_score:.4f}")

# In ra một số thông tin về mô hình SGD
print(f"\n--- THÔNG TIN MÔ HÌNH ---")
print(f"Số lượng epochs thực tế: {sgd_model.n_iter_}")
print(f"Số lượng features: {sgd_model.n_features_in_}")

# (Tùy chọn) Xem trọng số của các features
# Các trọng số này cho biết đóng góp của từng feature đến dự đoán
# print(f"Top 10 features có trọng số cao nhất:")
# feature_importance = np.abs(sgd_model.coef_[0])
# top_indices = np.argsort(feature_importance)[-10:]
# for idx in reversed(top_indices):
#     print(f"  Feature {idx}: {feature_importance[idx]:.6f}")

1. Tạo mô hình SGD Regressor...
2. Thiết lập thông số (Hyperparameters)...
3. Bắt đầu huấn luyện...


KeyboardInterrupt: 